# Word-Embedding Baseline and Text CNN for Text Classification

**Goal:** build a word-embedding baseline and a Text CNN for a text classification task.
**Dataset:** a labelled text classification dataset (198-example validation split).
**Approach:** a GloVe-embedding + logistic-regression baseline, and a Text CNN with three parallel Conv1d filters (kernel sizes 3, 5, 7) and global max-pooling.
**Headline result:** the Text CNN reached 97.01% test accuracy, well above the 90.40% validation accuracy of the GloVe + logistic-regression baseline. See `README.md` and `../../docs/RESULTS.md`.

*Built as part of Monash University FIT5215 Deep Learning (2026).*


Author: Rahul Bhatt

## Section 1: Fundamentals in RNNs

Implements a multi-timestep recurrent neural network by hand, operating on a 3D input tensor of shape `[batch_size, seq_len, input_size]`, for a classification task.

In [10]:
import torch

We declare the relevant variables.

In [12]:
input_size = 5
seq_len = 4
batch_size = 8
hidden_size = 3
num_classes = 3

We create random inputs (i.e., `inputs`) and random labels (i.e., `random_labels`).

In [14]:
inputs = torch.randn(batch_size, seq_len, input_size)
random_labels = torch.randint(0, num_classes, (batch_size,))
print(inputs.shape)
print(random_labels)

torch.Size([8, 4, 5])
tensor([1, 1, 1, 2, 1, 1, 0, 2])


Declares the RNN's model parameters: the input-to-hidden matrix $U$ (`[input_size, hidden_size]`), the hidden-to-hidden matrix $W$ (`[hidden_size, hidden_size]`), the hidden-to-output matrix $V$ (`[hidden_size, num_classes]`), and the bias vectors $b$ and $c$ for the hidden state and logits respectively.

In [16]:
#Insert your code here
U = torch.randn(input_size, hidden_size, requires_grad=True)
W = torch.randn(hidden_size, hidden_size, requires_grad=True)
b = torch.zeros(hidden_size, requires_grad=True)
V = torch.randn(hidden_size, num_classes, requires_grad=True)
c = torch.zeros(num_classes, requires_grad=True)
print(U.shape, W.shape, b.shape, V.shape, c.shape)

torch.Size([5, 3]) torch.Size([3, 3]) torch.Size([3]) torch.Size([3, 3]) torch.Size([3])


Computes `hiddens`, a 3D tensor of shape `[batch_size, seq_len, hidden_size]`, by applying the standard RNN recurrence at each timestep.

In [18]:
#Initialize hiddens for update.
# Simple RNN: h_t = tanh(x_t @ U + h_{t-1} @ W + b)
h_prev = torch.zeros(batch_size, hidden_size)
hidden_list = []
for t in range(seq_len):
    x_t = inputs[:, t, :]
    h_t = torch.tanh(x_t @ U + h_prev @ W + b)
    hidden_list.append(h_t)
    h_prev = h_t
hiddens = torch.stack(hidden_list, dim=1)  # [batch_size, seq_len, hidden_size]

print(hiddens.shape)
print(hiddens)

torch.Size([8, 4, 3])
tensor([[[ 0.8964,  0.8658,  0.0229],
         [ 0.9987, -0.8410, -0.9457],
         [ 0.9506, -0.7351, -0.3064],
         [ 0.9556, -0.9963, -0.9997]],

        [[ 0.9669,  0.9947,  0.9998],
         [-0.4873,  1.0000, -0.9975],
         [-0.7339,  0.9782,  0.4109],
         [-0.8404, -0.8491,  0.0699]],

        [[-0.4956, -0.9977,  0.3056],
         [ 0.9982, -1.0000,  0.9999],
         [ 0.9999, -1.0000,  0.9995],
         [ 0.9935, -0.9834,  0.9377]],

        [[-0.9499, -0.7654, -0.9969],
         [ 0.9570, -0.9902,  0.9959],
         [ 0.7957, -0.9932,  0.9914],
         [-0.4865, -0.4327,  0.8817]],

        [[ 0.9975,  0.9999, -0.9145],
         [-0.8063,  0.9440,  0.4004],
         [-1.0000,  0.9492, -0.9973],
         [ 0.6816,  0.3072, -0.9922]],

        [[ 0.9624, -0.9986, -0.5478],
         [ 0.8784, -0.9999, -0.8676],
         [ 0.9997, -0.9995,  0.9708],
         [ 0.9607, -0.6968,  0.9089]],

        [[ 0.9521, -0.9970, -1.0000],
         [ 0.997

Computes the classification logits from the final hidden state (`[batch_size, hidden_size]`) of the RNN.

In [20]:
last_hidden = hiddens[:, -1, :]  # [batch_size, hidden_size]
logits = last_hidden @ V + c
print(logits.shape)
print(logits)

torch.Size([8, 3])
tensor([[ 1.0142,  0.2752, -0.6884],
        [-0.6195,  2.6758,  2.3406],
        [-2.7357,  1.6621,  1.6806],
        [-1.9437,  2.0599,  2.3163],
        [ 2.0280, -2.0927, -2.4637],
        [-2.4588,  1.0898,  1.2229],
        [ 1.3292,  1.8820,  0.9547],
        [-2.8637,  1.7621,  1.8046]], grad_fn=<AddBackward0>)


Computes the cross-entropy loss between the predicted logits and the labels using PyTorch's built-in loss function.

In [22]:
#Insert your code here
criterion = torch.nn.CrossEntropyLoss()
loss = criterion(logits, random_labels)
print(loss)

tensor(1.2409, grad_fn=<NllLossBackward0>)


Performs backpropagation to compute the gradients of the loss with respect to the model parameters, using PyTorch's automatic differentiation.

In [24]:
#Insert your code here
loss.backward()
print("U.grad:\n", U.grad)
print("W.grad:\n", W.grad)
print("b.grad:\n", b.grad)
print("V.grad:\n", V.grad)
print("c.grad:\n", c.grad)

U.grad:
 tensor([[-0.0523, -0.1167, -0.0150],
        [-0.0162, -0.3052, -0.0129],
        [ 0.0663,  0.2064, -0.0188],
        [-0.0842, -0.2880,  0.0084],
        [-0.0075,  0.0264,  0.0047]])
W.grad:
 tensor([[-0.0979, -0.3386, -0.0078],
        [ 0.0061,  0.2957,  0.0058],
        [-0.1011, -0.3090,  0.0148]])
b.grad:
 tensor([0.0196, 0.3204, 0.0107])
V.grad:
 tensor([[ 0.2110, -0.2587,  0.0477],
        [ 0.0423,  0.0606, -0.1029],
        [-0.1037,  0.1317, -0.0280]])
c.grad:
 tensor([ 0.1151, -0.2182,  0.1031])


Manually updates the model parameters with a single SGD step, using a learning rate of $\eta = 0.1$.

In [26]:
#Insert your code here
eta = 0.1
with torch.no_grad():
    U -= eta * U.grad
    W -= eta * W.grad
    b -= eta * b.grad
    V -= eta * V.grad
    c -= eta * c.grad

print("Updated U:\n", U)
print("Updated W:\n", W)
print("Updated b:\n", b)
print("Updated V:\n", V)
print("Updated c:\n", c)

Updated U:
 tensor([[-1.0492,  0.5331, -1.9005],
        [-0.6499,  1.3157,  0.8168],
        [ 0.6724,  1.4856, -0.3211],
        [-1.4478,  0.2489,  0.3768],
        [ 0.3915, -0.6891, -0.9255]], requires_grad=True)
Updated W:
 tensor([[ 0.6224, -0.1732,  0.2591],
        [-0.8642,  0.7122, -1.7353],
        [ 0.7277, -0.2490,  0.0238]], requires_grad=True)
Updated b:
 tensor([-0.0020, -0.0320, -0.0011], requires_grad=True)
Updated V:
 tensor([[-0.2036, -1.0315, -1.0749],
        [ 0.7464, -2.0493, -1.5839],
        [-1.9266,  0.7370,  1.2571]], requires_grad=True)
Updated c:
 tensor([-0.0115,  0.0218, -0.0103], requires_grad=True)


## Section 2: Deep Learning for Sequential Data

### Set random seeds

We start with importing PyTorch and NumPy and setting random seeds for PyTorch and NumPy. You can use any seeds you prefer.

In [30]:
import os
import torch
import random
import requests
import pandas as pd
import numpy as np
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
from transformers import BertTokenizer
import os
from six.moves.urllib.request import urlretrieve
from sklearn import preprocessing
import matplotlib.pyplot as plt
plt.style.use('ggplot')

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [31]:
def seed_all(seed=1029):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
seed_all(seed=1234)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Download and preprocess the data

The dataset we use for this assignment is a question classification dataset for which the training set consists of $5,500$ questions belonging to 6 coarse question categories including:
- abbreviation (ABBR),
- entity (ENTY),
- description (DESC),
- human (HUM),
- location (LOC) and
- numeric (NUM).

In this assignment, we will utilize a subset of this dataset, containing $2,000$ questions for training and validation. We will use 80% of those 2000 questions for trainning and the rest for validation.


Preprocessing data is a crucial initial step in any machine learning or deep learning project. The *TextDataManager* class simplifies the process by providing functionalities to download and preprocess data specifically designed for the subsequent questions in this assignment. It is highly recommended to gain a comprehensive understanding of the class's functionality by **carefully reading** the content provided in the *TextDataManager* class before proceeding to answer the questions.

In [35]:
class DataManager:
    """
    This class manages and preprocesses a simple text dataset for a sentence classification task.

    Attributes:
        verbose (bool): Controls verbosity for printing information during data processing.
        max_sentence_len (int): The maximum length of a sentence in the dataset.
        str_questions (list): A list to store the string representations of the questions in the dataset.
        str_labels (list): A list to store the string representations of the labels in the dataset.
        numeral_labels (list): A list to store the numerical representations of the labels in the dataset.
        numeral_data (list): A list to store the numerical representations of the questions in the dataset.
        random_state (int): Seed value for random number generation to ensure reproducibility.
            Set this value to a specific integer to reproduce the same random sequence every time. Defaults to 6789.
        random (np.random.RandomState): Random number generator object initialized with the given random_state.
            It is used for various random operations in the class.

    Methods:
        maybe_download(dir_name, file_name, url, verbose=True):
            Downloads a file from a given URL if it does not exist in the specified directory.
            The directory and file are created if they do not exist.

        read_data(dir_name, file_names):
            Reads data from files in a directory, preprocesses it, and computes the maximum sentence length.
            Each file is expected to contain rows in the format "<label>:<question>".
            The labels and questions are stored as string representations.

        manipulate_data():
            Performs data manipulation by tokenizing, numericalizing, and padding the text data.
            The questions are tokenized and converted into numerical sequences using a tokenizer.
            The sequences are padded or truncated to the maximum sequence length.

        train_valid_test_split(train_ratio=0.9):
            Splits the data into training, validation, and test sets based on a given ratio.
            The data is randomly shuffled, and the specified ratio is used to determine the size of the training set.
            The string questions, numerical data, and numerical labels are split accordingly.
            TensorFlow `Dataset` objects are created for the training and validation sets.


    """

    def __init__(self, verbose=True, random_state=6789):
        self.verbose = verbose
        self.max_sentence_len = 0
        self.str_questions = list()
        self.str_labels = list()
        self.numeral_labels = list()
        self.maxlen = None
        self.numeral_data = list()
        self.random_state = random_state
        self.random = np.random.RandomState(random_state)

    @staticmethod
    def maybe_download(dir_name, file_name, url, verbose=True):
        if not os.path.exists(dir_name):
            os.mkdir(dir_name)
        if not os.path.exists(os.path.join(dir_name, file_name)):
            urlretrieve(url + file_name, os.path.join(dir_name, file_name))
        if verbose:
            print("Downloaded successfully {}".format(file_name))

    def read_data(self, dir_name, file_names):
        self.str_questions = list()
        self.str_labels = list()
        for file_name in file_names:
            file_path= os.path.join(dir_name, file_name)
            with open(file_path, "r", encoding="latin-1") as f:
                for row in f:
                    row_str = row.split(":")
                    label, question = row_str[0], row_str[1]
                    question = question.lower()
                    self.str_labels.append(label)
                    self.str_questions.append(question[0:-1])
                    if self.max_sentence_len < len(self.str_questions[-1]):
                        self.max_sentence_len = len(self.str_questions[-1])

        # turns labels into numbers
        le = preprocessing.LabelEncoder()
        le.fit(self.str_labels)
        self.numeral_labels = np.array(le.transform(self.str_labels))
        self.str_classes = le.classes_
        self.num_classes = len(self.str_classes)
        if self.verbose:
            print("\nSample questions and corresponding labels... \n")
            print(self.str_questions[0:5])
            print(self.str_labels[0:5])

    def manipulate_data(self):
        self.tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
        vocab = self.tokenizer.get_vocab()
        self.word2idx = {w: i for i, w in enumerate(vocab)}
        self.idx2word = {i:w for w,i in self.word2idx.items()}
        self.vocab_size = len(self.word2idx)

        token_ids = []
        num_seqs = []
        for text in self.str_questions:  # iterate over the list of text
          text_seqs = self.tokenizer.tokenize(str(text))  # tokenize each text individually
          # Convert tokens to IDs
          token_ids = self.tokenizer.convert_tokens_to_ids(text_seqs)
          # Convert token IDs to a tensor of indices using your word2idx mapping
          seq_tensor = torch.LongTensor(token_ids)
          num_seqs.append(seq_tensor)  # append the tensor for each sequence

        # Pad the sequences and create a tensor
        if num_seqs:
          self.numeral_data = pad_sequence(num_seqs, batch_first=True)  # Pads to max length of the sequences
          self.num_sentences, self.maxlen = self.numeral_data.shape

    def train_valid_test_split(self, train_ratio=0.8, test_ratio = 0.1):
        train_size = int(self.num_sentences*train_ratio) +1
        test_size = int(self.num_sentences*test_ratio) +1
        valid_size = self.num_sentences - (train_size + test_size)
        data_indices = list(range(self.num_sentences))
        random.shuffle(data_indices)
        self.train_str_questions = [self.str_questions[i] for i in data_indices[:train_size]]
        self.train_numeral_labels = self.numeral_labels[data_indices[:train_size]]
        train_set_data = self.numeral_data[data_indices[:train_size]]
        train_set_labels = self.numeral_labels[data_indices[:train_size]]
        train_set_labels = torch.from_numpy(train_set_labels)
        train_set = torch.utils.data.TensorDataset(train_set_data, train_set_labels)
        self.test_str_questions = [self.str_questions[i] for i in data_indices[-test_size:]]
        self.test_numeral_labels = self.numeral_labels[data_indices[-test_size:]]
        test_set_data = self.numeral_data[data_indices[-test_size:]]
        test_set_labels = self.numeral_labels[data_indices[-test_size:]]
        test_set_labels = torch.from_numpy(test_set_labels)
        test_set = torch.utils.data.TensorDataset(test_set_data, test_set_labels)
        self.valid_str_questions = [self.str_questions[i] for i in data_indices[train_size:-test_size]]
        self.valid_numeral_labels = self.numeral_labels[data_indices[train_size:-test_size]]
        valid_set_data = self.numeral_data[data_indices[train_size:-test_size]]
        valid_set_labels = self.numeral_labels[data_indices[train_size:-test_size]]
        valid_set_labels = torch.from_numpy(valid_set_labels)
        valid_set = torch.utils.data.TensorDataset(valid_set_data, valid_set_labels)
        self.train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
        self.test_loader = DataLoader(test_set, batch_size=64, shuffle=False)
        self.valid_loader = DataLoader(valid_set, batch_size=64, shuffle=False)

In [36]:
print('Loading data...')
DataManager.maybe_download("data", "train_2000.label", "http://cogcomp.org/Data/QA/QC/")

dm = DataManager()
dm.read_data("data/", ["train_2000.label"])

Loading data...


Downloaded successfully train_2000.label

Sample questions and corresponding labels... 

['manner how did serfdom develop in and then leave russia ?', 'cremat what films featured the character popeye doyle ?', "manner how can i find a list of celebrities ' real names ?", 'animal what fowl grabs the spotlight after the chinese year of the monkey ?', 'exp what is the full form of .com ?']
['DESC', 'ENTY', 'DESC', 'ENTY', 'ABBR']


In [37]:
dm.manipulate_data()
dm.train_valid_test_split(train_ratio=0.8, test_ratio = 0.1)

In [38]:
for x, y in dm.train_loader:
    print(x.shape, y.shape)
    break

torch.Size([64, 36]) torch.Size([64])


## Part 1: Word-Embedding Baseline for Text Classification

In [40]:
import gensim.downloader as api
from gensim.models import Word2Vec
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score
import numpy as np

### GloVe Word Vectors

Loads the pretrained `glove-wiki-gigaword-100` embedding model and implements `get_word_vector`, which maps a word to its 100-dimensional GloVe vector, returning a zero vector for words outside the vocabulary.

In [42]:
word2vect = api.load("glove-wiki-gigaword-100")

[--------------------------------------------------] 0.0% 0.0/128.1MB downloaded

[--------------------------------------------------] 0.0% 0.0/128.1MB downloaded

[--------------------------------------------------] 0.0% 0.0/128.1MB downloaded

[--------------------------------------------------] 0.0% 0.0/128.1MB downloaded

[--------------------------------------------------] 0.0% 0.0/128.1MB downloaded

[--------------------------------------------------] 0.0% 0.0/128.1MB downloaded

... (16,386 outputs left out)

[=================================================-] 100.0% 128.1/128.1MB downloaded

[=================================================-] 100.0% 128.1/128.1MB downloaded

[=================================================-] 100.0% 128.1/128.1MB downloaded

[==================================================] 100.0% 128.1/128.1MB downloaded

In [43]:
def get_word_vector(word, model):
    try:
        vector = model[word]
    except:
        vector = np.zeros(100)
    return vector

### Sentence Vectors via Weighted Averaging

Implements `get_sentence_vector(sentence, important_score, model)`, which represents a sentence as a weighted average of its words' GloVe vectors. A softmax over per-word `important_score` values produces the weights; when no scores are given, the function falls back to a plain (equal-weight) average of the word vectors.

In [45]:
def get_sentence_vector(sentence, important_score=None, model= None):
    #Insert your code here
    words = sentence.split() if isinstance(sentence, str) else list(sentence)
    if len(words) == 0:
        return np.zeros(100)

    if important_score is None:
        important_score = np.ones(len(words))
    else:
        important_score = np.asarray(important_score, dtype=np.float64)

    # important_weight = softmax(important_score)
    shifted = important_score - np.max(important_score)
    exp_scores = np.exp(shifted)
    important_weight = exp_scores / np.sum(exp_scores)

    word_vectors = np.vstack([get_word_vector(word, model) for word in words])
    feature_vector = np.sum(important_weight[:, np.newaxis] * word_vectors, axis=0)
    return feature_vector

### Building Feature Matrices

Converts the training and validation question sets into NumPy feature matrices (`X_train`, `X_valid`) and label arrays (`y_train`, `y_valid`) using the sentence-vector function above, with per-word importance scores decaying across each sentence.

In [47]:
print("Transform training set to feature vectors...")
X_train_list = []
for sentence in dm.train_str_questions:
    words = sentence.split()
    important_score = [0.9 ** i for i in range(len(words))]
    X_train_list.append(get_sentence_vector(sentence, important_score=important_score, model=word2vect))
X_train = np.array(X_train_list)
y_train = dm.train_numeral_labels
print("X_train shape:", X_train.shape, "y_train shape:", y_train.shape)

Transform training set to feature vectors...
X_train shape: (1601, 100) y_train shape: (1601,)


In [48]:
print("Transform validation set to feature vectors...")
X_valid_list = []
for sentence in dm.valid_str_questions:
    words = sentence.split()
    important_score = [0.9 ** i for i in range(len(words))]
    X_valid_list.append(get_sentence_vector(sentence, important_score=important_score, model=word2vect))
X_valid = np.array(X_valid_list)
y_valid = dm.valid_numeral_labels
print("X_valid shape:", X_valid.shape, "y_valid shape:", y_valid.shape)

Transform validation set to feature vectors...
X_valid shape: (198, 100) y_valid shape: (198,)


### Feature Scaling

Scales the training and validation feature vectors to the range (-1, 1) using scikit-learn's `MinMaxScaler`.

In [50]:
#Insert your code
scaler = MinMaxScaler(feature_range=(-1, 1))
X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
print("Scaled X_train range:", X_train.min(), X_train.max())
print("Scaled X_valid range:", X_valid.min(), X_valid.max())



Scaled X_train range: -1.0 1.0000000000000004
Scaled X_valid range: -1.218093457636006 1.2510827014064667


### Logistic Regression Baseline

Trains a logistic regression classifier on the GloVe-derived features and evaluates it on the validation set using standard classification metrics.

In [52]:
from sklearn.linear_model import LogisticRegression
#Insert your code
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)
y_pred = logreg.predict(X_valid)


In [53]:
from sklearn import metrics
#Insert your code
print("Validation accuracy:", metrics.accuracy_score(y_valid, y_pred))
print("\nClassification report:\n", metrics.classification_report(y_valid, y_pred))
print("Confusion matrix:\n", metrics.confusion_matrix(y_valid, y_pred))


Validation accuracy: 0.9040404040404041

Classification report:
               precision    recall  f1-score   support

           0       1.00      0.50      0.67         2
           1       0.91      0.86      0.89        36
           2       0.89      0.87      0.88        54
           3       0.96      1.00      0.98        52
           4       0.84      0.90      0.87        29
           5       0.88      0.88      0.88        25

    accuracy                           0.90       198
   macro avg       0.91      0.83      0.86       198
weighted avg       0.90      0.90      0.90       198

Confusion matrix:
 [[ 1  1  0  0  0  0]
 [ 0 31  2  1  1  1]
 [ 0  2 47  1  2  2]
 [ 0  0  0 52  0  0]
 [ 0  0  3  0 26  0]
 [ 0  0  1  0  2 22]]


We now declare the `BaseTrainer` class, which will be used later to train the subsequent deep learning models for text data.

In [55]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class BaseTrainer:
    def __init__(self, model, criterion, optimizer, train_loader, val_loader):
        self.model = model
        self.criterion = criterion  #the loss function
        self.optimizer = optimizer  #the optimizer
        self.train_loader = train_loader  #the train loader
        self.val_loader = val_loader  #the valid loader

    #the function to train the model in many epochs
    def fit(self, num_epochs):
        self.num_batches = len(self.train_loader)

        for epoch in range(num_epochs):
            print(f'Epoch {epoch + 1}/{num_epochs}')
            train_loss, train_accuracy = self.train_one_epoch()
            val_loss, val_accuracy = self.validate_one_epoch()
            print(
                f'{self.num_batches}/{self.num_batches} - train_loss: {train_loss:.4f} - train_accuracy: {train_accuracy*100:.4f}% \
                - val_loss: {val_loss:.4f} - val_accuracy: {val_accuracy*100:.4f}%')

    #train in one epoch, return the train_acc, train_loss
    def train_one_epoch(self):
        self.model.train()
        running_loss, correct, total = 0.0, 0, 0
        for i, data in enumerate(self.train_loader):
            inputs, labels = data
            inputs, labels = inputs.to(device), labels.to(device)
            self.optimizer.zero_grad()
            outputs = self.model(inputs)
            loss = self.criterion(outputs, labels)
            loss.backward()
            self.optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        train_accuracy = correct / total
        train_loss = running_loss / self.num_batches
        return train_loss, train_accuracy

    #evaluate on a loader and return the loss and accuracy
    def evaluate(self, loader):
        self.model.eval()
        loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for data in loader:
                inputs, labels = data
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = self.model(inputs)
                loss = self.criterion(outputs, labels)
                loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        accuracy = correct / total
        loss = loss / len(self.val_loader)
        return loss, accuracy

    #return the val_acc, val_loss, be called at the end of each epoch
    def validate_one_epoch(self):
      val_loss, val_accuracy = self.evaluate(self.val_loader)
      return val_loss, val_accuracy

## Part 2: Text CNN for Sequence Modeling and Neural Embedding

### Text CNN Architecture

Implements a Text CNN for sentence classification (Kim, 2014). Input tokens are embedded, then passed through three parallel `Conv1D` layers with kernel sizes 3, 5, and 7; each is global-max-pooled, the three pooled vectors are concatenated, and a final dense layer produces the class logits.

In [58]:
import torch.nn as nn
import torch.nn.functional as F

#You can modify the code if you want but need to keep the skeleton
class TextCNN(torch.nn.Module):
    def __init__(self, embed_size= 128, state_size=16, data_manager=None):
        super().__init__()
        self.data_manager = data_manager
        self.embed_size = embed_size
        self.state_size = state_size
        #declare the necessary layers here
        self.embed = nn.Embedding(self.data_manager.vocab_size, self.embed_size)
        self.conv1d_1 = nn.Conv1d(in_channels=self.embed_size, out_channels=self.state_size, kernel_size=3, padding=1)
        self.conv1d_2 = nn.Conv1d(in_channels=self.embed_size, out_channels=self.state_size, kernel_size=5, padding=2)
        self.conv1d_3 = nn.Conv1d(in_channels=self.embed_size, out_channels=self.state_size, kernel_size=7, padding=3)
        self.fc = nn.Linear(state_size*3, self.data_manager.num_classes)

    def forward(self, x):
        e = self.embed(x)
        #permute x before applying Conv1D
        e= e.permute(0,2,1)

        #applying Conv1D
        h1 = F.relu(self.conv1d_1(e))
        h2 = F.relu(self.conv1d_2(e))
        h3 = F.relu(self.conv1d_3(e))

        #apply GlobalMaxPool
        h1 = F.max_pool1d(h1, kernel_size=h1.shape[-1]).squeeze(-1)
        h2 = F.max_pool1d(h2, kernel_size=h2.shape[-1]).squeeze(-1)
        h3 = F.max_pool1d(h3, kernel_size=h3.shape[-1]).squeeze(-1)

        h = torch.cat([h1, h2, h3], dim=1)
        h = self.fc(h)
        return h

We declare `text_cnn` and train on several epochs (e.g., `50 epochs`).


In [60]:
text_cnn = TextCNN(data_manager=dm).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(text_cnn.parameters(), lr=0.001)
trainer = BaseTrainer(model=text_cnn, criterion=criterion, optimizer=optimizer, train_loader=dm.train_loader, val_loader=dm.valid_loader)
trainer.fit(num_epochs=50)

Epoch 1/50


26/26 - train_loss: 1.5324 - train_accuracy: 41.4116%                 - val_loss: 0.5336 - val_accuracy: 74.2424%
Epoch 2/50


26/26 - train_loss: 0.6891 - train_accuracy: 84.0725%                 - val_loss: 0.2533 - val_accuracy: 83.3333%
Epoch 3/50


26/26 - train_loss: 0.3977 - train_accuracy: 92.0050%                 - val_loss: 0.1887 - val_accuracy: 92.4242%
Epoch 4/50


26/26 - train_loss: 0.2177 - train_accuracy: 95.5653%                 - val_loss: 0.1841 - val_accuracy: 92.4242%
Epoch 5/50


26/26 - train_loss: 0.1460 - train_accuracy: 97.6889%                 - val_loss: 0.1562 - val_accuracy: 93.4343%
Epoch 6/50


... (41 outputs left out)

26/26 - train_loss: 0.0010 - train_accuracy: 100.0000%                 - val_loss: 0.0261 - val_accuracy: 96.4646%
Epoch 48/50


26/26 - train_loss: 0.0012 - train_accuracy: 100.0000%                 - val_loss: 0.0264 - val_accuracy: 96.4646%
Epoch 49/50


26/26 - train_loss: 0.0010 - train_accuracy: 100.0000%                 - val_loss: 0.0266 - val_accuracy: 96.4646%
Epoch 50/50


26/26 - train_loss: 0.0009 - train_accuracy: 100.0000%                 - val_loss: 0.0235 - val_accuracy: 96.4646%


We evaluate the trained model on the testing set.

In [62]:
test_loss, test_acc = trainer.evaluate(dm.test_loader)
print(f'test_loss: {test_loss:.4f} - test_accuracy: {test_acc*100:.4f}%')

test_loss: 0.2371 - test_accuracy: 97.0149%
